# El mapa de su literatura
**Diplomado en IA para la Investigación · Módulo 3 · Sesión 11** · Alonso Ramírez Hernández · CENES–UPTC

Este cuaderno trae los artículos de su tema desde **OpenAlex** (una base abierta), convierte cada resumen en un **embedding** (un vector que representa su significado) y los dibuja como un **mapa**: lo que trata de lo mismo queda cerca. Al final **audita la representación**: en qué idiomas, de qué países y de qué años es lo que encontró.

**Cómo se usa:** cambie su tema en el paso 1 y corra todo (Entorno de ejecución → Ejecutar todas). Tarda dos o tres minutos.

**Lo que se lleva:** la figura del mapa para su estación de literatura (`02_literatura`) y tres líneas para su bitácora.

## Paso 1 · Su tema, en español y en inglés

In [ ]:
CONSULTA_ES = "violencia y rendimiento escolar"      # su tema en español
CONSULTA_EN = "violence and school achievement"     # su tema en inglés
POR_IDIOMA  = 150    # cuántos artículos trae por cada consulta (máximo 200)
DESDE       = 1990   # año desde el que busca

## Paso 2 · Preparar las herramientas

In [ ]:
import subprocess, sys
try:
    import sentence_transformers
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=False)
import requests, numpy as np, pandas as pd, re, collections
import plotly.express as px, matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfVectorizer
print("Listo")

## Paso 3 · Traer los artículos de OpenAlex

In [ ]:
CAMPOS = "id,display_name,publication_year,language,abstract_inverted_index,authorships,cited_by_count,primary_location,open_access"
def resumen(inv):
    if not inv: return ""
    pos = {}
    for palabra, lugares in inv.items():
        for p in lugares: pos[p] = palabra
    return " ".join(pos[i] for i in sorted(pos))
def traer(consulta, n):
    filas, pagina = [], 1
    while len(filas) < n:
        r = requests.get("https://api.openalex.org/works", timeout=60, params={
            "search": consulta, "filter": f"from_publication_date:{DESDE}-01-01",
            "per-page": min(100, n), "page": pagina, "select": CAMPOS})
        r.raise_for_status(); d = r.json()
        if not d["results"]: break
        filas += d["results"]; pagina += 1
    return filas[:n]
crudo = traer(CONSULTA_ES, POR_IDIOMA) + traer(CONSULTA_EN, POR_IDIOMA)
vistos, filas = set(), []
for w in crudo:
    if w["id"] in vistos: continue
    vistos.add(w["id"])
    paises = sorted({c for a in (w.get("authorships") or []) for c in (a.get("countries") or [])})
    fuente = ((w.get("primary_location") or {}).get("source") or {}).get("display_name") or ""
    filas.append({"titulo": w.get("display_name") or "", "anio": w.get("publication_year"),
                  "idioma": w.get("language") or "sin dato", "resumen": resumen(w.get("abstract_inverted_index")),
                  "paises": paises, "citas": w.get("cited_by_count", 0), "revista": fuente,
                  "acceso_abierto": (w.get("open_access") or {}).get("is_oa", False)})
df = pd.DataFrame(filas)
df["texto"] = (df["titulo"] + ". " + df["resumen"]).str.strip()
df["con_resumen"] = df["resumen"].str.len() > 0
print(f"{len(df)} artículos distintos · {df['con_resumen'].sum()} con resumen · {(~df['con_resumen']).sum()} solo con título")
if len(df) < 40: print("⚠️ Muy pocos artículos para un mapa: amplíe su consulta (menos palabras, sinónimos) y vuelva a correr desde el paso 1.")
df.head()

## Paso 4 · Los embeddings: cada artículo se vuelve un vector
El modelo es multilingüe: un artículo en español y otro en inglés sobre lo mismo quedan cerca.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    X = modelo.encode(df["texto"].tolist(), show_progress_bar=True, normalize_embeddings=True)
    METODO = "embeddings multilingües (paraphrase-multilingual-MiniLM-L12-v2)"
except Exception as e:
    from sklearn.decomposition import TruncatedSVD
    print("Sin el modelo de embeddings; uso TF-IDF + SVD:", str(e)[:120])
    X = TruncatedSVD(n_components=100, random_state=1).fit_transform(TfidfVectorizer(min_df=2).fit_transform(df["texto"]))
    METODO = "TF-IDF + SVD (respaldo)"
print(METODO, "·", X.shape)

## Paso 5 · Los grupos y el mapa

In [ ]:
K = int(np.clip(round(len(df) / 30), 4, 8))
df["grupo"] = KMeans(n_clusters=K, n_init=10, random_state=1).fit_predict(X)
VACIAS = set("""de la el los las y en del a un una por para con se que su sus al como es o lo más entre sobre this the of and in to a for on with by from as an at is are was were be that these which using study results
we our their its it between effect effects analysis data based also has have not been can may new two three""".split())
tf = TfidfVectorizer(max_features=4000, token_pattern=r"(?u)\b[a-záéíóúñü]{3,}\b")
M = tf.fit_transform(df["texto"].str.lower()); voc = np.array(tf.get_feature_names_out())
nombres = {}
for g in range(K):
    fila = np.asarray(M[(df["grupo"] == g).values].mean(axis=0)).ravel()
    top = [w for w in voc[fila.argsort()[::-1]] if w not in VACIAS][:4]
    nombres[g] = f"{g+1} · " + ", ".join(top)
df["tema"] = df["grupo"].map(nombres)
xy = TSNE(n_components=2, perplexity=min(30, max(5, len(df)//5)), random_state=1, init="pca").fit_transform(X)
df["x"], df["y"] = xy[:, 0], xy[:, 1]
fig = px.scatter(df, x="x", y="y", color="tema", symbol="idioma", hover_data={"titulo": True, "anio": True, "revista": True, "x": False, "y": False},
                 title=f"El mapa de su literatura · {len(df)} artículos en {K} grupos", height=650)
fig.update_layout(xaxis_visible=False, yaxis_visible=False, legend_title_text="Grupo (palabras más típicas)")
fig.show()
print(df.groupby("tema").size().rename("artículos"))

## Paso 6 · ¿Quién está en su mapa y quién no?
La auditoría de representación: idioma, región y años de lo que encontró.

In [ ]:
LATAM = set("AR BO BR CL CO CR CU DO EC SV GT HN MX NI PA PY PE PR UY VE JM TT HT BZ GY SR BS BB".split())
df["con_latam"] = df["paises"].apply(lambda p: bool(set(p) & LATAM))
df["con_colombia"] = df["paises"].apply(lambda p: "CO" in p)
n = len(df)
idiomas = df["idioma"].value_counts()
print(f"De {n} artículos:")
for k, v in idiomas.head(5).items(): print(f"  · {v} en «{k}» ({v/n:.0%})")
print(f"  · {df['con_latam'].sum()} con al menos un autor en América Latina ({df['con_latam'].mean():.0%}); {df['con_colombia'].sum()} con autor en Colombia")
print(f"  · {df['acceso_abierto'].mean():.0%} en acceso abierto · mediana del año: {int(df['anio'].median())}")
paises = collections.Counter(c for p in df["paises"] for c in p)
fig2, ax = plt.subplots(1, 3, figsize=(15, 4))
idiomas.head(6).sort_values().plot.barh(ax=ax[0], color="#9aa5b8"); ax[0].set_title("Idioma")
pd.Series(dict(paises.most_common(10))).sort_values().plot.barh(ax=ax[1], color=["#1B44CC" if c in LATAM else "#9aa5b8" for c in pd.Series(dict(paises.most_common(10))).sort_values().index]); ax[1].set_title("Países de los autores (azul: América Latina)")
df["anio"].plot.hist(ax=ax[2], bins=20, color="#9aa5b8"); ax[2].set_title("Año de publicación")
plt.tight_layout(); plt.show()
print("\nPor grupo: ¿dónde está América Latina?")
print((df.groupby("tema")["con_latam"].mean()*100).round().astype(int).astype(str) + " %")

## Paso 7 · La figura para su carpeta
Con la ficha de la figura de la sesión 9: el título dice el hallazgo, la nota dice la fuente, la n y lo que quedó fuera.

In [ ]:
fig3, ax = plt.subplots(figsize=(11, 7))
for g, sub in df.groupby("grupo"):
    ax.scatter(sub["x"], sub["y"], s=22, alpha=.8, label=nombres[g])
ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values(): s.set_visible(False)
TITULO = "El mapa de mi literatura"   # ← cámbielo por su hallazgo: ¿qué muestra el mapa?
ax.set_title(TITULO, loc="left", fontsize=15, fontweight="bold")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1), frameon=False, fontsize=9)
import textwrap
NOTA = (f"Fuente: OpenAlex, consultas «{CONSULTA_ES}» y «{CONSULTA_EN}», desde {DESDE}. n = {len(df)} artículos ({(~df['con_resumen']).sum()} solo con título). "
        f"Mapa: {METODO} y t-SNE. {df['con_latam'].mean():.0%} con al menos un autor en América Latina.")
fig3.text(0.01, 0.01, textwrap.fill(NOTA, 150), fontsize=8, color="#555")
plt.tight_layout(rect=(0, 0.04, 1, 1)); fig3.savefig("S11_mapa_literatura.png", dpi=200); plt.show()
df.drop(columns=["texto"]).to_csv("S11_mapa_literatura.csv", index=False)
print("Guardé S11_mapa_literatura.png y S11_mapa_literatura.csv: descárguelos (panel de archivos, a la izquierda) y póngalos en 02_literatura.")

## Paso 8 · Lea su mapa (tres líneas para su bitácora)
1. **¿Dónde está su pregunta?** ¿En qué grupo cae, o entre cuáles? Una zona con pocos puntos puede ser su hueco o una búsqueda mal hecha.
2. **¿Quién no está?** ¿Qué idiomas, regiones o años faltan? ¿Su tabla de evidencia tiene el mismo sesgo?
3. **¿Qué cambiaría?** Una palabra más en la consulta, otra base (SciELO, LILACS) o un criterio de inclusión.

**Ojo:** el mapa depende de la consulta. Es una exploración, no una revisión sistemática: los artículos no están verificados uno por uno.